# EDA - Exploration des donnees de detection de fraude

Ce notebook explore le dataset Kaggle "Credit Card Fraud Detection" avant l'entrainement des modeles.

Objectifs :
- Comprendre la structure des donnees
- Visualiser le desequilibre des classes (legitime vs fraude)
- Comparer les montants des transactions fraude vs legitime
- Regarder les correlations entre features

## 1. Chargement et aperçu du dataset

In [ ]:
import sys
sys.path.append("..")  # pour pouvoir importer src/ depuis le notebook

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import KAGGLE_DATASET_PATH

sns.set_style("whitegrid")

In [ ]:
# Chargement du dataset
df = pd.read_csv(KAGGLE_DATASET_PATH)

print(f"Shape : {df.shape}")
df.head()

In [ ]:
# Infos generales : types de colonnes, valeurs manquantes
df.info()

In [ ]:
# Statistiques descriptives de base
df.describe()

## 2. Distribution des classes

On s'attend a un dataset tres desequilibre : tres peu de fraudes par rapport aux transactions legitimes.

In [ ]:
class_counts = df["Class"].value_counts()
pct_fraud = class_counts[1] / class_counts.sum() * 100

print(f"Transactions legitimes (0) : {class_counts[0]}")
print(f"Transactions fraude (1)    : {class_counts[1]}")
print(f"Pourcentage de fraude      : {pct_fraud:.3f}%")

plt.figure(figsize=(6, 4))
sns.countplot(x="Class", data=df)
plt.title("Distribution des classes (0 = legitime, 1 = fraude)")
plt.xlabel("Classe")
plt.ylabel("Nombre de transactions")
plt.yscale("log")  # echelle log car la classe 1 est ecrasee sinon
plt.show()

## 3. Distribution des montants : fraude vs legitime

In [ ]:
fraud = df[df["Class"] == 1]
legit = df[df["Class"] == 0]

print("Statistiques du montant (Amount) - transactions legitimes :")
print(legit["Amount"].describe())

print("\nStatistiques du montant (Amount) - transactions fraude :")
print(fraud["Amount"].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(legit["Amount"], bins=50, color="steelblue")
axes[0].set_title("Montants - transactions legitimes")
axes[0].set_xlabel("Montant")
axes[0].set_ylabel("Frequence")

axes[1].hist(fraud["Amount"], bins=50, color="crimson")
axes[1].set_title("Montants - transactions fraude")
axes[1].set_xlabel("Montant")
axes[1].set_ylabel("Frequence")

plt.tight_layout()
plt.show()

## 4. Correlation des features

Les colonnes V1..V28 sont issues d'une transformation PCA (anonymisation), donc theoriquement peu correlees entre elles. On verifie aussi leur correlation avec la cible `Class`.

In [ ]:
corr = df.corr()

plt.figure(figsize=(14, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, cbar=True)
plt.title("Matrice de correlation - toutes les features")
plt.show()

In [ ]:
# Correlation de chaque feature avec la cible Class, triee par valeur absolue
corr_with_target = corr["Class"].drop("Class").sort_values(key=abs, ascending=False)

plt.figure(figsize=(8, 8))
corr_with_target.plot(kind="barh")
plt.title("Correlation de chaque feature avec Class (fraude)")
plt.xlabel("Coefficient de correlation")
plt.gca().invert_yaxis()
plt.show()

## Resume

Apres cette exploration, on sait que :
- Le dataset est extremement desequilibre (~0.17% de fraudes) -> il faudra utiliser SMOTE.
- Les montants de fraude ont une distribution differente des montants legitimes.
- Certaines features V* sont plus correlees a la fraude que d'autres.

L'entrainement des modeles se fait dans `src/train.py`. Une fois execute, il affiche :

```
Modele sauvegarde avec AUC-ROC = X.XX
```